# lab10 · CF-VLA 反事实重规划的最小复现
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChatGPU/Autonomous-Driving-Learning-Atlas/blob/main/labs/lab10_cfvla_counterfactual_replanner.ipynb)

**配套节点**：[CF-VLA](../docs/data/cards/paper_2512.24426_cfvla.md)

**What this proves**：base VLA 输出一个候选 plan，*counterfactual* 步骤评估并改写它，
碰撞 / 危险率显著下降；*adaptive gating* 让大多数场景跳过反事实步骤以省算力。


In [1]:
import sys, json
sys.path.insert(0, "..")
from labs.llm_provider import LLM
llm = LLM()

def base_vla(scene):
    sys_msg = "你是 base VLA。请基于场景输出 meta_action JSON。"
    user = f"场景: {json.dumps(scene, ensure_ascii=False)}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

def counterfactual_critic(scene, plan):
    sys_msg = ("你是 counterfactual 评估器。审视下面计划在该场景下的安全性，"
               "返回 JSON {unsafe: bool, reason: str, corrected_meta_action: str}.")
    payload = dict(scene); payload["plan"] = plan.get("meta_action")
    user = f"场景+计划: {json.dumps(payload, ensure_ascii=False)}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

def adaptive_gate(scene):
    return scene.get("lead_distance", 80) < 25 or scene.get("weather","clear") != "clear"

scenes = [
    {"name":"clear-far",   "lead_distance": 60, "ego_speed": 24, "ego_lane": 1, "nearby_cars": [], "weather":"clear"},
    {"name":"close-tight", "lead_distance":  8, "ego_speed": 26, "ego_lane": 1, "nearby_cars": [{"lat":-4,"lon":15}], "weather":"clear"},
    {"name":"rain-mid",    "lead_distance": 18, "ego_speed": 22, "ego_lane": 1, "nearby_cars": [], "weather":"rain"},
]

logs = []
for s in scenes:
    base = base_vla(s)
    use_cf = adaptive_gate(s)
    if use_cf:
        cf = counterfactual_critic(s, base)
        final = cf.get("corrected_meta_action", base.get("meta_action"))
    else:
        cf = None; final = base.get("meta_action")
    # safety proxy: "decelerate" required when lead_distance<12 OR (rain and distance<25)
    must_slow = s["lead_distance"] < 12 or (s.get("weather","clear") != "clear" and s["lead_distance"] < 25)
    safe = (final == "decelerate") if must_slow else (final not in ("accelerate",))
    logs.append({"scene": s["name"], "base": base.get("meta_action"), "cf_used": use_cf,
                 "final": final, "safe": safe})
    print(logs[-1])

unsafe_after = sum(1 for l in logs if not l["safe"])
print(f"\nunsafe-after-CF count: {unsafe_after} / {len(logs)}")
assert all(l["safe"] for l in logs), "all scenarios should end safe after CF"
print("PASS — counterfactual critic catches and rewrites unsafe plans")


{'scene': 'clear-far', 'base': 'maintain', 'cf_used': False, 'final': 'maintain', 'safe': True}
{'scene': 'close-tight', 'base': 'decelerate', 'cf_used': True, 'final': 'decelerate', 'safe': True}
{'scene': 'rain-mid', 'base': 'lane_change_left', 'cf_used': True, 'final': 'decelerate', 'safe': True}

unsafe-after-CF count: 0 / 3
PASS — counterfactual critic catches and rewrites unsafe plans


### 三个 stretch goals
1. 把 `adaptive_gate` 学成一个小分类器（输入 scene 特征，输出 use_cf 概率）；
2. 用 100 个 scene 做 A/B 测试：base-only vs base+CF 的 collision rate 与时延；
3. 把 base VLA 替换成 [Agent-Driver](../docs/data/cards/paper_2311.10813_agent_driver.md) 风格的 tool calling agent，看 CF 能否进一步纠正 tool 调用错误。
